In [ ]:
import pandas as pd
from collections import defaultdict

# ============================================================
# CONFIG
# ============================================================

METADATA_CSV = "streetview_metadata.csv"
ACCIDENTS_CSV = "collisions_final.csv"
OUTPUT_CSV = "streetview_metadata_with_collision.csv"

metadata = pd.read_csv(METADATA_CSV)
accidents = pd.read_csv(ACCIDENTS_CSV)

# ============================================================
# Build lookup: (lat, lon) -> list of collision_ids
# ============================================================

lookup = defaultdict(list)

for _, row in accidents.iterrows():
    key = (row["latitude"], row["longitude"])
    lookup[key].append(row["collision_id"])

# ============================================================
# Assign one collision_id per metadata row
# ============================================================

collision_ids = []

for _, row in metadata.iterrows():
    key = (row["requested_latitude"], row["requested_longitude"])

    if lookup[key]:
        collision_ids.append(lookup[key].pop(0))
    else:
        collision_ids.append(pd.NA)

metadata.insert(1, "collision_id", collision_ids)

# ============================================================
# SAVE
# ============================================================

metadata.to_csv(OUTPUT_CSV, index=False)

print(f"Metadata rows : {len(metadata):,}")
print(f"Matched rows  : {metadata['collision_id'].notna().sum():,}")
print(f"Unmatched     : {metadata['collision_id'].isna().sum():,}")

print(f"\nSaved to {OUTPUT_CSV}")